# AI as Judge

[G-Eval](https://deepeval.com/docs/metrics-llm-evals) is a framework that uses LLM as a judge to evaluate LLM outputs. The evaluation can be based on any criteria. G-Eval is implemented by a library called [DeepEval](https://deepeval.com/) which includes a broader set of tests.


In [1]:
%load_ext dotenv
%dotenv ../../05_src/.secrets

In [6]:
from openai import OpenAI
import os

document_folder = "../../05_src/documents/"
chesterton_file = "chesterton.txt"
file_path = os.path.join(document_folder, chesterton_file)

with open(file_path, "r", encoding="utf-8") as f:
    chesterton_text = f.read()

In [5]:
instructions = "You are an helpful assistant that summarizes works of fiction with a quirky and bubbly approach."
PROMPT = """
    Summarize the following story in at most four paragraphs. Please include all key characters and plot points.
    <story>
    {story}
    </story>
    In addition to the summary, add an introduction paragraph where you greet the reader and a conclusion where you share an opinion about the story.
"""

In [7]:
import os
client = OpenAI(base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1', 
                api_key='any value',
                default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')})
response = client.responses.create(
    model="gpt-4o-mini",
    instructions=instructions,
    input=[
        {"role": "user", 
         "content": PROMPT.format(story=chesterton_text)}
    ],
    temperature=1.2
)

In [8]:
response.output_text

'Hello, marvelous reader! 🌟 Let me whisk you away to the curious world of "The Innocence of Father Brown" by G.K. Chesterton, where whimsy meets mystery in delightful harmony! Our dapper little priest, Father Brown, navigates a tapestry of tales, solving crimes that unfold in unexpected ways—most notably with his astute, unassuming nature and uncanny knack for getting to the heart of intrigue.\n\nIn “The Blue Cross,” the story kicks off with police detective Valentin chasing the notorious criminal Flambeau through a blend of holiday cheer and intrigue, only to find himself matched in wits by the inconspicuous Father Brown, who proves that the meek can indeed uncover the most profound truths. Then in “The Secret Garden,” we dive into the hidden depths of character as Brown unveils the moral complexities of human nature within a gritty narrative involving justice and forgiveness amidst economic despair. The whimsy continues in “The Queer Feet” where the detective duo confronts puzzling m

# Answer Relevancy

The answer relevancy metric evaluates how relevant the actual output of the LLM app is compared to the provided input. This metric is self-explaining in the sense that the output includes a reason for the metric score.

The metric is calculated as:

$$
AnswerRelevancy=\frac{NumberRelevantStatements}{TotalStatements}
$$

Reference: [Answer Relevancy](https://deepeval.com/docs/metrics-answer-relevancy). 

In [9]:
from deepeval import evaluate
from deepeval.metrics import AnswerRelevancyMetric
from deepeval.test_case import LLMTestCase
from deepeval.models import GPTModel

model = GPTModel(
    model="gpt-4o-mini",
    temperature=0,
    # api_key='any value',
    default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')},
    base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1',
)

metric = AnswerRelevancyMetric(
    threshold=0.7,
    include_reason=True,
    model=model,
    
)

test_case = LLMTestCase(
    input=PROMPT.format(story=chesterton_text),
    actual_output=response.output_text,
    
)

In [10]:
metric.measure(test_case)

Output()

1.0

In [11]:
from IPython.display import display, Markdown
display(Markdown(f'**Score**: {metric.score}'))
display(Markdown(f'**Reason**: {metric.reason}'))

**Score**: 1.0

**Reason**: The score is 1.00 because the response directly addresses the request for a summary of the story, including all key characters and plot points, without any irrelevant statements.

# Other Metrics

Other useful metric functions include:

+ [Faithfulness](https://deepeval.com/docs/metrics-faithfulness): evaluates whether the `actual_output` factually aligns with the contents of  `retrieval_context`. 
+ [Contextual Precision](https://deepeval.com/docs/metrics-contextual-precision): evaluates whether nodes in your `retrieval_context` that are relevant to the given input are ranked higher than irrelevant ones. 
+ [Contextual Recall](https://deepeval.com/docs/metrics-contextual-recall): evaluates the extent of which the retrieval_context aligns with the expected_output. 
+ [Contextual Relevancy](https://deepeval.com/docs/metrics-contextual-relevancy): evaluates the overall relevance of the information presented in your retrieval_context for a given input. 

# G-Eval

[G-Eval](https://deepeval.com/docs/metrics-llm-evals) is a framework that uses LLM-as-a-judge with chain-of-thoughts (CoT) to evaluate LLM outputs based on ANY custom criteria. The G-Eval metric is the most versatile type of metric deepeval offers.

In [12]:
instructions = "You are an helpful assistant that specializes in works of fiction."
PROMPT = """
    Based on the story below, answer the question provided.
    <story>
    {story}
    </story>
    <question>
    Who is the main antagonist in the story and what motivates their actions?
    </question>
"""

In [13]:
response = client.responses.create(
    model="gpt-4o-mini",
    instructions=instructions,
    input=[
        {"role": "user", 
         "content": PROMPT.format(story=chesterton_text)}
    ],
    temperature=0.7
)

In [14]:
response.output_text

'The main antagonist in "The Innocence of Father Brown" is often portrayed as Flambeau, a master criminal who embodies the archetype of the clever and charismatic thief. However, in several of the stories, including "The Blue Cross," Flambeau ultimately undergoes a transformation and becomes more of an ally to Father Brown rather than a true antagonist.\n\nIn "The Blue Cross," Flambeau\'s motivations stem from his desire for challenge and the thrill of the chase. He is intrigued by the idea of outsmarting Father Brown, who represents the force of good and justice. Flambeau\'s actions are driven by a mix of criminal ambition and a fascination with the detective\'s intellect. \n\nThroughout the collection, Flambeau\'s character evolves, reflecting a complex interplay between crime, morality, and redemption. Ultimately, his motivations reveal a deeper search for meaning beyond mere theft, as he becomes increasingly drawn to the moral clarity that Father Brown represents.'

## Evaluation Criteria

The most straightforward way to establish a metric is by using a single criteria.

In [15]:
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCaseParams

correctness_metric = GEval(
    name="Correctness",
    criteria="Determine whether the actual output is factually correct based on the context.",
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
    model=model,
)

In [16]:
test_case = LLMTestCase(
    input=PROMPT.format(story=chesterton_text),
    actual_output=response.output_text
)
evaluate(test_cases=[test_case], metrics=[correctness_metric])

✨ You're running DeepEval's latest Correctness [GEval] Metric! (using gpt-4o-mini, strict=False, 
async_mode=True)...

Output()



Metrics Summary

  - ✅ Correctness [GEval] (score: 0.7863388359327709, threshold: 0.5, strict: False, evaluation model: gpt-4o-mini, reason: The response accurately identifies Flambeau as the main antagonist in the story, highlighting his motivations of challenge and thrill in outsmarting Father Brown. It also notes Flambeau's character evolution and his eventual alignment with Father Brown, which reflects a nuanced understanding of the narrative. However, the response could be improved by explicitly mentioning other potential antagonists in the broader context of the collection, as well as providing more detail on the specific story referenced., error: None)

For test case:

  - input: 
    Based on the story below, answer the question provided.
    <story>
    ﻿The Project Gutenberg eBook of The innocence of Father Brown
    
This ebook is for the use of anyone anywhere in the United States and
most other parts of the world at no cost and with almost no restrictions
whatsoever. You

✓ Evaluation completed 🎉! (time taken: 15.77s | token cost: 0.0166737 USD)
» Test Results (1 total tests):
   » Pass Rate: 100.0% | Passed: 1 | Failed: 0

 ================================================================================ 

» What to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

EvaluationResult(test_results=[TestResult(name='test_case_0', success=True, metrics_data=[MetricData(name='Correctness [GEval]', threshold=0.5, success=True, score=0.7863388359327709, reason="The response accurately identifies Flambeau as the main antagonist in the story, highlighting his motivations of challenge and thrill in outsmarting Father Brown. It also notes Flambeau's character evolution and his eventual alignment with Father Brown, which reflects a nuanced understanding of the narrative. However, the response could be improved by explicitly mentioning other potential antagonists in the broader context of the collection, as well as providing more detail on the specific story referenced.", strict_mode=False, evaluation_model='gpt-4o-mini', error=None, evaluation_cost=0.0166737, verbose_logs='Criteria:\nDetermine whether the actual output is factually correct based on the context. \n \nEvaluation Steps:\n[\n    "1. Identify the context provided in the input to establish the fact

## Evaluation Steps 

G-Eval is flexible in many ways: notice that we can establish an evaluation criteria or a set of evaluation steps, that can help in guiding the model to follow specific steps to perform the evaluation.

In [17]:
...

correctness_metric = GEval(
    name="Correctness",
    evaluation_steps=[
        "Check whether the facts in 'actual output' contradicts any facts in 'input'",
        "You should also heavily penalize omission of detail",
        "Vague language, or contradicting OPINIONS, are not OK"
    ],
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
    model=model,
)

In [18]:
test_case = LLMTestCase(
    input=PROMPT.format(story=chesterton_text),
    actual_output=response.output_text
)
result = evaluate(test_cases=[test_case], metrics=[correctness_metric])

✨ You're running DeepEval's latest Correctness [GEval] Metric! (using gpt-4o-mini, strict=False, 
async_mode=True)...

Output()



Metrics Summary

  - ✅ Correctness [GEval] (score: 0.6928246400330816, threshold: 0.5, strict: False, evaluation model: gpt-4o-mini, reason: The response identifies Flambeau as the main antagonist, which aligns with the story's portrayal of his character. It accurately describes his motivations as a mix of criminal ambition and a desire for intellectual challenge against Father Brown. However, it could have provided more specific examples from the text to illustrate Flambeau's transformation and the nuances of his character, which would strengthen the analysis. Additionally, the response could clarify that Flambeau's role as an antagonist is not consistent throughout the entire collection, as he becomes more of an ally in later stories., error: None)

For test case:

  - input: 
    Based on the story below, answer the question provided.
    <story>
    ﻿The Project Gutenberg eBook of The innocence of Father Brown
    
This ebook is for the use of anyone anywhere in the United States

✓ Evaluation completed 🎉! (time taken: 18.47s | token cost: 0.016600349999999996 USD)
» Test Results (1 total tests):
   » Pass Rate: 100.0% | Passed: 1 | Failed: 0

 ================================================================================ 

» What to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

In [19]:
result.model_dump()

{'test_results': [{'name': 'test_case_0',
   'success': True,
   'metrics_data': [{'name': 'Correctness [GEval]',
     'threshold': 0.5,
     'success': True,
     'score': 0.6928246400330816,
     'reason': "The response identifies Flambeau as the main antagonist, which aligns with the story's portrayal of his character. It accurately describes his motivations as a mix of criminal ambition and a desire for intellectual challenge against Father Brown. However, it could have provided more specific examples from the text to illustrate Flambeau's transformation and the nuances of his character, which would strengthen the analysis. Additionally, the response could clarify that Flambeau's role as an antagonist is not consistent throughout the entire collection, as he becomes more of an ally in later stories.",
     'strict_mode': False,
     'evaluation_model': 'gpt-4o-mini',
     'error': None,
     'evaluation_cost': 0.016600349999999996,
     'verbose_logs': 'Criteria:\nNone \n \nEvaluat